<h2><b>计算机高等教育通用教材</b></h2>
<h2>机器学习 Machine learning</h2>
<hr>
<h5>第一部分：监督学习 supervised learning</h5>
<h5>第四章：分类与规则提取 决策树 Decision Tree</h5>
<hr>
<h3><b>实验四：基于决策树预测泰坦尼克号幸存乘客</b></h3>
<hr>
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html'>查看DecisionTreeClassifier源代码(sklearn)</a><br>
<br>

> **适合人群** ：你已经跨过了线性回归和逻辑回归的门槛，学会了让机器去算“权重（w）”。但现实中，人类做决策往往不是在脑子里算一堆小数的加减乘除，而是做一系列的“如果-那么”判断（比如：如果下雨，我就带伞；如果是周末，我就睡懒觉）。
> 本章，我们将带你学习一种完全模仿人类思考方式、极其透明且极具解释性的白盒算法——决策树。
<hr>

#### 第0步：测试python与虚拟环境

In [ ]:
print("Hello Decision Tree!")
import pip
print("Pip version:", pip.__version__)

<hr><hr>

#### 第一步：import库 & 导入数据
<hr>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn import model_selection
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

In [ ]:
# 教材中使用了经典的泰坦尼克号沉船数据集（titanic.csv）。
# 为了让你能够一键运行这份代码，我们在这里通过严格的概率控制，
# 生成一份包含 500 名乘客的逼真模拟数据。
# 字段包括：乘客编号、船票级别、性别、年龄、兄妹配偶个数、是否幸存。

np.random.seed(42)
n_samples = 500

passenger_ids = np.arange(1, n_samples + 1)
pclass = np.random.choice([1, 2, 3], n_samples, p=[0.25, 0.25, 0.5]) # 3等舱人最多
sex = np.random.choice(['male', 'female'], n_samples)
age = np.random.normal(30, 15, n_samples)
age = np.clip(age, 1, 80) # 年龄限制在1到80岁
sibsp = np.random.choice([0, 1, 2, 3, 4], n_samples, p=[0.6, 0.2, 0.1, 0.05, 0.05])

# 制造缺失值：现实世界的数据很少是完美的，我们故意让 50 个人的年龄数据丢失（变成 NaN）
missing_idx = np.random.choice(n_samples, 50, replace=False)
age[missing_idx] = np.nan

# 制造隐藏的生存逻辑：女士优先、头等舱优先、儿童优先
survival_score = np.zeros(n_samples)
survival_score += np.where(sex == 'female', 2.0, -1.0)
survival_score += np.where(pclass == 1, 1.5, np.where(pclass == 2, 0.5, -1.5))
# 这里忽略 NaN 的比较警告
with np.errstate(invalid='ignore'):
    survival_score += np.where(age < 12, 1.5, 0) 

probs = 1 / (1 + np.exp(-survival_score))
survived = (np.random.rand(n_samples) < probs).astype(int)

df = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Pclass': pclass,
    'Sex': sex,
    'Age': age,
    'SibSp': sibsp,
    'Survived': survived
})

print(f'数据集大小: {df.shape}')
df.head(10)

<hr><hr>

#### 第二步：查看数据的基本信息 与 空值处理（极其重要）
<hr>

In [ ]:
df.info()
# 仔细看输出结果，你会发现除了 Age 这一列是 450 non-null，其他列都是 500 non-null。
# 这意味着 Age 列有 50 个空值 (Missing values)。

In [ ]:
# 统计各字段空值数量
null_result = df.isnull().sum()
print("各字段空值统计：\n", null_result)

# 计算年龄字段缺失率
age_null_per = null_result["Age"] / df.shape[0]
print(f"年龄字段缺失率 = {(age_null_per * 100):.2f}%")

In [ ]:
# 遇到空值怎么办？
# 机器学习模型是个极其严谨的数学系统，如果里面混进去一个 NaN（Not a Number），整个系统就会崩溃报错。
# 我们有两种做法：1. 把这 50 个人删掉。2. 猜一个数字填进去。
# 删掉太可惜了，数据很宝贵。所以我们用“填充（Fillna）”法。

# 我们可以用所有乘客年龄的平均值（mean）、中位数（median）或者众数（mode）来填充。
# 这里我们选用平均值来填充缺失的年龄。

age_mean = df['Age'].mean()
print(f"计算出所有已知乘客的平均年龄为：{age_mean:.1f}岁")

# 将 NaN 替换为平均值
df['Age'].fillna(age_mean, inplace=True)

# 再次验证空值是否填充完成
print("填充后各字段空值统计：\n", df.isnull().sum())
# 看到全变成 0，我们就安全了。

<hr><hr>

#### 第三步：数据探查与交叉频数表
<hr>

In [ ]:
# 为了人类阅读方便，我们把数字映射成汉字。但请记住，模型只认识数字。
# 所以我们单独复制一份 DataFrame 用于探查，原始的 df 留给模型。
df_explore = df.copy()
df_explore['Pclass'] = df_explore['Pclass'].map({1: '头等舱', 2: '二等舱', 3: '三等舱'})
df_explore['Sex'] = df_explore['Sex'].map({'male': '男', 'female': '女'})
df_explore['Survived'] = df_explore['Survived'].map({0: '死亡', 1: '幸存'})

# 制作特征与标签的【交叉频数表】
# 交叉频数表是极其强大的工具，它能一眼看穿两个分类变量之间有没有关联。

print("【性别 与 存亡比较结果】")
cross_sex = pd.crosstab(df_explore['Sex'], df_explore['Survived'], margins=True)
print(cross_sex)

print("\n【船票级别 与 存亡比较结果】")
cross_pclass = pd.crosstab(df_explore['Pclass'], df_explore['Survived'], margins=True)
print(cross_pclass)

# 结果分析：
# 从交叉表中可以极其清晰地看到，女性的幸存人数远大于死亡人数，而男性恰好相反。
# 头等舱的幸存率也明显高于三等舱。
# 决策树在训练时，一定会敏锐地捕捉到这些强烈的信号。

<hr><hr>

#### 第四步：数据转换与特征降维
<hr>

In [ ]:
# 1. 字符型特征转数值型
# 模型不认识 'male' 和 'female'，我们需要把它变成 0 和 1。
df['Sex'] = df['Sex'].map({'male': 1, 'female': 0})

# 2. 降维：删除无关特征
# PassengerId 只是一个乘客编号，就像你的学号一样。学号是单数还是双数，跟这艘船沉没时你能不能活下来，毫无因果关系。
# 如果不删除它，模型可能会强行去寻找编号和存活的关联（比如发现编号为2、5、8的人活了，就以为这是个规律），这会导致严重的误判。
# 所以我们要把它丢弃。

X = df[['Pclass', 'Sex', 'Age', 'SibSp']].values
y = df['Survived'].values

print("降维后，输入特征 X 的模样 (前两行):")
print(X[:2])

# 注意：我们在上一章说逻辑回归必须做“数据标准化（StandardScaler）”。
# 在决策树里，我们需要做标准化吗？
# 答案是：完全不需要！
# 决策树的底层逻辑不是算权重 w，而是像切西瓜一样在特征上画线（比如：年龄是否大于 12 岁？）。
# 无论年龄是 12 还是 1200，决策树只关心“大于还是小于某个阈值”，它对数据的绝对大小和量纲完全免疫！

<hr><hr>

#### 第五步：模型训练
<hr>

In [ ]:
# 1. 切分考卷
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, 
    random_state=42, 
    test_size=0.2
)

# 2. 呼叫主角：决策树分类器
# max_depth=3 是什么意思？它限制了这棵树最多只能往下分三个层级（问三个问题）。
# 为什么要限制它？我们在后面的【拓展提高】部分会详细讲解这个极其重要的概念。
dt_model = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)

print("决策树模型训练完毕！")

<hr><hr>

#### 第六步：模型评估 与 规则提取（白盒之美）
<hr>

In [ ]:
# 让模型做期末考试卷
y_test_pred = dt_model.predict(X_test)

# 计算准确率
acc = accuracy_score(y_test, y_test_pred)
print(f"决策树对泰坦尼克号生存预测的准确率 = {(acc * 100):.2f}%")

In [ ]:
# 配置中文字体用于树状图展示
zh_fonts = [f.name for f in fm.fontManager.ttflist 
            if any(kw in f.name for kw in ['Hei', 'Song', 'CJK', 'Chinese', 'SC', 'TC', 'Gothic', 'SimHei'])]
if zh_fonts:
    plt.rcParams['font.family'] = zh_fonts[0]
plt.rcParams['axes.unicode_minus'] = False 

# 决策树最大的优势：它是“白盒模型”。
# 线性回归和神经网络脑子里是一堆让人看不懂的权重和算子，但决策树脑子里是一套非常清晰的业务规则。
# 我们可以直接把这套规则画出来看！

plt.figure(figsize=(15, 8))
plot_tree(
    dt_model, 
    feature_names=['船票级别', '性别(1=男,0=女)', '年龄', '亲属数'],
    class_names=['死亡', '幸存'],
    filled=True,      # 给节点上色
    rounded=True,     # 圆角矩形
    fontsize=10
)
plt.title("机器学到的泰坦尼克号生存法则 (决策树可视化)")
plt.show()

# 如何看懂这棵树？
# 1. 顶端的根节点问了第一个问题：性别 <= 0.5 吗？（因为我们把女设为0，男设为1，这个问题等价于“是女性吗？”）
# 2. 如果是女性（走左边分支 True），它接着问：船票级别 <= 2.5 吗？（是头等舱或二等舱吗？）
# 3. 顺着路径走下去，最终停在的叶子节点，上面写的 class 就是模型给出的预测结果。
# 颜色越偏向蓝色，幸存概率越大；越偏向橙色，死亡概率越大。

In [ ]:
# 提取特征重要性
# 决策树非常聪明，被它越早选中来提问的特征，往往越重要。
importance = dt_model.feature_importances_
feature_names = ['船票级别', '性别', '年龄', '亲属数']

print("\n【特征重要性排序】")
for name, imp in zip(feature_names, importance):
    print(f"{name:.<10} {imp*100:.2f}%")

# 显然，性别是生死存亡最核心的特征。

<hr><hr>

#### 第七步：【拓展提高】为什么我们要“剪枝”？
<hr>

在刚才训练模型时，我们传了一个参数 `max_depth=3`。如果你把它去掉，让机器放飞自我去训练，会发生什么？

回答这个问题之前，我们必须理解机器学习中一个永远的幽灵：**过拟合（Overfitting）**。

假设你在一家大公司当 HR，你拿到了一份包含了 1000 个离职员工的数据，你想建一棵决策树来预测未来谁会离职。
如果你不给决策树任何限制，它为了追求在现有的 1000 个人里达到 100% 的准确率，它会不断地分叉、不断地问问题，直到把每一个人都单独区分开来。

它最后长出来的规则可能是这样的：
1. 如果薪水 < 5000，且
2. 如果加班时长 > 40小时，且
3. 如果年龄 > 35岁，且
4. **如果是双鱼座，且**
5. **如果左脸颊有一颗痣，且**
6. **如果今天早上吃了包子** ----> **判定：会离职**。

发现问题了吗？这棵树为了强行在这 1000 个人里找规律，把很多纯粹的**巧合与噪声**（比如左脸颊有痣）当成了真理。
它死记硬背了所有数据，导致它错综复杂，长得像一棵参天大树。
这种树在训练集上的分数是完美的 100 分，但在现实世界（测试集）中，新来一个左脸颊有痣的员工，它直接预言人家要离职，准确率极其惨淡。

**这就叫过拟合：学得太死板，丧失了泛化能力（举一反三的能力）。**

##### 什么是剪枝（Pruning）？

顾名思义，剪枝就是拿一把大剪刀，把这棵长得太过茂盛、问了太多无聊问题的树的枝丫给咔嚓掉。

1. **预剪枝（Pre-pruning）**：
就是防患于未然。在树还没长大的时候，我们就给它定下规矩。
- 比如我们设置 `max_depth=3`，意思是：“你最多只能问三个最关键的问题！问完必须给我结论，不许再往下钻牛角尖了！”
- 或者是 `min_samples_split=20`：“如果这一波人只剩下不到 20 个了，你就停止分叉，直接用少数服从多数给我个结论。”
这就是我们刚才代码里做的事情，它极其有效。

2. **后剪枝（Post-pruning）**：
就是先让它放飞自我长成参天大树，然后我们自下而上地去检查。如果发现剪掉最底下的某根树枝，模型在测试集上的表现并没有变差，那说明这根树枝纯属废话，咔嚓剪掉。

**一句话总结剪枝：懂得放手。不要试图完美地拟合每一个特例，抓住宏观的普遍规律才是好模型。**

<br><hr>

#### 第八步：【拓展提高】三个臭皮匠顶个诸葛亮 —— 随机森林（Random Forest）
<hr>

既然我们知道了单棵决策树容易钻牛角尖（过拟合），那有没有一种办法，既能保留决策树好懂、不用标准化的优点，又能彻底解决它不稳定的毛病？

恭喜你，接触到了机器学习中最伟大、最实用、在各类竞赛中屡拿冠军的神级算法：**随机森林（Random Forest）**。

##### 什么是随机森林？

如果把一棵决策树比作一个医生，那么随机森林就是一个拥有 100 个医生的**专家会诊委员会**。

面对一个病人（一条新数据），我们不让一个医生说了算。我们让这 100 个医生每个人都对病人进行诊断，然后大家**举手表决**。
如果有 80 个医生说没病，20 个医生说有病，那么最终结论就是没病（少数服从多数）。

这种把许多个弱小的模型集合起来，发挥出强大力量的方法，叫做**集成学习（Ensemble Learning）**。

##### 为什么叫“随机”森林？

这里有一个极度深刻的哲学问题：
如果这 100 个医生，他们在同一所医科大学读同一本教材，拜同一个老师为师。那么面对同一个病人，他们的判断一定是一模一样的。
既然大家都一样，那 100 个医生投票和 1 个医生投票有什么区别？毫无区别！

为了让举手表决有意义，我们必须保证这 100 个医生**每个人都有自己独特的见解和偏好**。
随机森林是如何制造这种差异的呢？全靠“随机”这两个字：

1. **数据随机（Bootstrap 抽样）**：
我们不把全部的病人资料给每一个医生。假设我们有 1000 份病历，我们给 1 号医生随机抽 1000 份（有放回抽样，所以有些病历重复，有些病历根本没抽到）。给 2 号医生重新随机抽。
这样，每个医生见过的世面都不完全一样。

2. **特征随机（最精妙的一步）**：
假设一个病人有 20 项体检指标。如果允许所有医生看这 20 项指标，绝大多数医生建树的时候，第一步一定会看最明显的那个指标（比如血压）。大家长出来的树顶端全长得一样。
随机森林非常霸道，它规定：
- 1 号医生，我只允许你看身高、血糖、心率这 5 个指标来诊断。
- 2 号医生，我只允许你看体重、年龄、血脂这 5 个指标来诊断。

强行遮掉一部分特征，逼迫每个医生只能基于局部的视角去思考。
有些医生可能会得出错误的结论，但在最终 100 个医生共同举手表决时，**局部的偏见会被集体的智慧相互抵消，而真正的真理会沉淀下来。**

这就是随机森林极其强大的原因。它几乎不怎么需要调参，不会轻易陷入过拟合的深井，对噪声和异常值极其鲁棒。它是真正的工业级杀器。

In [ ]:
# 为了展示它的威力，我们用刚才同样的数据，来训练一个随机森林看看
from sklearn.ensemble import RandomForestClassifier

# n_estimators=100 意味着我们雇佣了 100 棵决策树（100 个医生）
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 我们不给它设置 max_depth，让每个医生自由发挥
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"\n单棵决策树的准确率: {(acc * 100):.2f}%")
print(f"百人专家团(随机森林)的准确率: {(rf_acc * 100):.2f}%")

# 往往你会发现，随机森林即便不费尽心思去“剪枝”，它的准确率和稳定性也会碾压单棵决策树。
# 这就是集体的力量。

#### 总结

| 概念 | 大白话解释 | 特点与注意事项 |
|------|------|------|
| **决策树** | 用一连串的 IF-ELSE 问题来做决定。 | 白盒模型，极其透明，像人类逻辑。**不需要数据标准化**。 |
| **缺失值处理** | 数据里有空洞（NaN）。 | 不能留空，视情况用平均值、中位数或众数（字符型）填补。 |
| **过拟合** | 钻牛角尖，死记硬背。把左脸有痣当成离职前兆。 | 训练集分数极高，新数据预测一塌糊涂。 |
| **剪枝** | 给树定规矩，不让它问太多无聊问题。 | `max_depth` (最大深度) 是最常用的预剪枝手段。懂得放手才是真理。 |
| **随机森林** | 100 棵结构各异的决策树一起举手表决。 | 集成学习的代表。通过“数据随机”和“特征随机”保证多样性，抗过拟合能力极强。 |

<br>

> **关键点**：这节课你学会了从最贴近人类直觉的“决策树”，进化到了能够利用集体智慧的“随机森林”。请细细体会“特征随机抽样”以打破常规偏见的设计哲学，这不仅仅是数学，更是深刻的工程智慧与社会学哲理。

<br>

<hr><hr>

## 实验四完成
<hr>

##### 此实验教材最近更新时间 2026年3月17日
<hr><hr>